# Feature Engineering

This notebook focuses on transforming telemetry data into actionable performance indicators. By calculating metrics for speed, braking, throttle application, and gear usage, the process creates a structured dataset that quantifies driving behavior, enabling a deeper understanding of track management.

##  Team and Grand Prix Selection
Define parameters for the current analysis, including the team, Grand Prix, and racing season. This scope ensures that the feature engineering process targets the correct, previously processed dataset.

In [1]:
team = "McLaren"
grand_prix = "British Grand Prix"
year = 2025

## Setup and Dependencies

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Fetch cleaned and interpolated telemetry data for both teammates. Loading pre-prepared data ensures that feature calculations are performed on standardized, high-quality inputs.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(year, grand_prix, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (99.92 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for British Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '63', '44', '16', '12', '87', '14', '10', '55', '22', '6', '23', '31', '30', '5', '18', '27', '43']



Processed data for 2025 British Grand Prix PIA and NOR found in cache.
Driver 1: PIA, Driver 2: NOR, Segment: Q3


## Compute Speed Metrics

Compute speed-related performance metrics such as: entry speed, minimum speed and exit speed throughout the corners. This analysis examines speed profiles relative to track turns to identify differences in corner entry and exit speeds, providing insights into raw pace and efficiency.

In [4]:
try:
    speed_metrics_1 = analyze_speed_metrics(lap_1, turns)
    speed_metrics_2 = analyze_speed_metrics(lap_2, turns)
except Exception as e:
    print(f"Speed metrics analysis failed: {e}")

print(f"Speed metrics for {driver_1}:\n\n{speed_metrics_1}\n")
print(f"Speed metrics for {driver_2}:\n\n{speed_metrics_2}")

Speed metrics for PIA:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  307.000000    303.056012  303.320998
1      2  307.000000    305.000002  305.000000
2      3  246.453030    123.005290  120.296060
3      4  134.594681     91.810026  139.443143
4      5  199.277188    204.046836  244.976047
5      6  254.356927    181.899089  183.771977
6      7  137.790197    119.082053  163.775500
7      8  269.072895    270.816955  288.276044
8      9  314.874033    305.531997  304.745055
9     10  308.000000    301.768084  300.608042
10    11  307.321987    296.235047  297.165989
11    12  295.812102    272.164997  270.631111
12    13  259.612182    232.381016  239.183878
13    14  246.225907    249.357000  270.116900
14    15  283.672332    250.881999  250.277044
15    16  264.543871    111.000179  112.712881
16    17  111.000179    111.243998  156.256499
17    18  188.075731    192.354008  238.356887

Speed metrics for NOR:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  3

## Compute Braking Zones

Identify and analyze braking zones throughout the lap. This section evaluates how each driver approaches braking points—specifically focusing on the start and end of braking events—to highlight differences in commitment and corner entry style registering information such as: braking duration, braking distance and braking point.

In [5]:
try:
    braking_zones_1 = analyze_braking_zones(lap_1, turns)
    braking_zones_2 = analyze_braking_zones(lap_2, turns)
except Exception as e:
    print(f"Braking zones analysis failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

Braking zones for PIA:

:     Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0      1       0.000000     0.000000         0.000000         0.000000   
1      2       0.000000     0.000000         0.000000         0.000000   
2      3     804.168619   859.628524         1.242995        55.459905   
3      4     952.061698   998.278286         1.324017        46.216587   
4      5       0.000000     0.000000         0.000000         0.000000   
5      6    1857.906810  1941.096667         1.260996        83.189857   
6      7    2061.259794  2116.719698         1.290996        55.459905   
7      8       0.000000     0.000000         0.000000         0.000000   
8      9       0.000000     0.000000         0.000000         0.000000   
9     10       0.000000     0.000000         0.000000         0.000000   
10    11       0.000000     0.000000         0.000000         0.000000   
11    12       0.000000     0.000000         0.000000         0.000000   
12    13    

## Compute Throttle Segments

Calculate throttle usage segments. This analysis determines when and how aggressively each driver applies the throttle coming out of corners, serving as a critical indicator of traction management and power deployment. Each segment registers: mean throttle, throttle standard deviation and full throttle percentage. 

In [6]:
try:
    throttle_segments_1 = analyze_throttle_segments(lap_1, turns)
    throttle_segments_2 = analyze_throttle_segments(lap_2, turns)
except Exception as e:
    print(f"Throttle segments analysis failed: {e}")


print(f"Throttle segments for {driver_1}:\n\n: {throttle_segments_1}\n")
print(f"Throttle segments for {driver_2}:\n\n: {throttle_segments_2}\n")

Throttle segments for PIA:

:     Segment  MeanThrottle  StdThrottle  FullThrottlePercentage
0         0     98.733428     0.436742              100.000000
1         1     98.699941     0.445873              100.000000
2         2     71.110426    41.280175               61.538462
3         3     19.329017    29.811428                0.000000
4         4     84.750176    25.779617               65.000000
5         5     86.170419    32.850839               86.250000
6         6     23.390520    31.042911                0.000000
7         7     94.834679    11.759678               83.783784
8         8     98.883483     0.305218              100.000000
9         9     98.992359     0.043775              100.000000
10       10     98.059500     0.092835              100.000000
11       11     85.259124    16.963975               53.333333
12       12     37.626586    26.257161                0.000000
13       13     94.667537     8.515566               80.000000
14       14     88.132098

## Compute Gear Shifts

Examine gear shift patterns across the lap registering: number of shifts, lowest gear, highest gear, minimum rpm, maximum rpm, mean rpn and rpm standard deviation. Comparing gear selection at different track segments reveals variations in driving technique, engine management, and how torque and traction are balanced.

In [7]:
try:
    gear_shifts_1 = analyze_gear_shifts(lap_1, turns)
    gear_shifts_2 = analyze_gear_shifts(lap_2, turns)
except Exception as e:
    print(f"Gear shifts analysis failed: {e}")

print(f"Gear shifts for {driver_1}:\n\n: {gear_shifts_1}\n")
print(f"Gear shifts for {driver_2}:\n\n: {gear_shifts_2}\n")

Gear shifts for PIA:

:     Segment  NumberOfShifts  LowestGear  HighestGear  MinimumRPM  MaximumRPM  \
0         0               1           7            8       10537       11637   
1         1               0           8            8       10711       10823   
2         2               1           5            8        9441       11280   
3         3               1           3            3        6812       10960   
4         4               2           3            5        6682       11454   
5         5               5           5            8       10345       11756   
6         6               1           4            4        7423       12019   
7         7               3           4            7        7634       11716   
8         8               1           7            8       10485       11624   
9         9               0           8            8       10538       10909   
10       10               0           8            8       10678       10853   
11       11     

## Cache Processed Data

Integrate the newly calculated metrics into the driver data structures and save the augmented dataset to the cache. This ensures that extracted features are persistent and readily available for subsequent performance comparisons and visualization steps.

In [8]:
lap_1["SpeedMetrics"] = speed_metrics_1
lap_1["BrakingZones"] = braking_zones_1 
lap_1["ThrottleSegments"] = throttle_segments_1
lap_1["GearShifts"] = gear_shifts_1


lap_2["SpeedMetrics"] = speed_metrics_2
lap_2["BrakingZones"] = braking_zones_2
lap_2["ThrottleSegments"] = throttle_segments_2
lap_2["GearShifts"] = gear_shifts_2

try:
    for dataframe_name in ["SpeedMetrics", "BrakingZones", "ThrottleSegments", "GearShifts"]:
        save_dataframe(lap_1, dataframe_name, year, grand_prix)
        save_dataframe(lap_2, dataframe_name, year, grand_prix)
except Exception as e:
    print(f"Failed to save dataframes: {e}")

try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_PIA.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_PIA.pkl
Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_NOR.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_british_grand_prix_Q3_NOR.pkl

Processed data for 2025 British Grand Prix saved to cache.
Driver 1: PIA, Driver 2: NOR, Segment: Q3
